In [ ]:
#3 – Očkování podle zemí
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# Načtení dat
df = pd.read_csv(r"D:\CodersLab\project_1_python.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df_last = df[df["date"] == df["date"].max()].copy()  # potřebuju poslední den

# Výpočet poměru očkování na obyvatele
df_last["vaccinations_per_capita"] = df_last["total_vaccinations"] / df_last["population"]

# Dash aplikace
app = Dash(__name__)

app.layout = html.Div([
    html.H1("COVID-19 Vaccination Dashboard"),

    html.Label("Select number of countries to display:"),
    dcc.Dropdown(
        id="n-filter",
        options=[{"label": str(i), "value": i} for i in [5, 10, 15, 20]],
        value=5
    ),

    html.Div([
        dcc.Graph(id="total-vaccinations-graph"),
        dcc.Graph(id="vaccinations-per-capita-graph")
    ], style={"display": "flex"})
])

@app.callback(
    [Output("total-vaccinations-graph", "figure"),
     Output("vaccinations-per-capita-graph", "figure")],
    [Input("n-filter", "value")]
)
def update_graphs(n):
    top_total = df_last.nlargest(n, "total_vaccinations")
    top_ratio = df_last.nlargest(n, "vaccinations_per_capita")

    fig_total = px.bar(
        top_total,
        x="location",
        y="total_vaccinations",
        title=f"Top {n} countries by total vaccinations"
    )

    fig_ratio = px.bar(
        top_ratio,
        x="location",
        y="vaccinations_per_capita",
        title=f"Top {n} countries by vaccinations per capita"
    )

    return fig_total, fig_ratio

if __name__ == "__main__":
    app.run(debug=True)
